# Analisis Data Eksperimen Pilot Study: Efektivitas Isolasi Cgroups
**Judul Penelitian**: Efektivitas Konfigurasi Isolasi Cgroups dalam Memitigasi *Noisy Neighbor* pada Lingkungan Berbasis *Container*  
**Peneliti**: Elgin Brian Wahyu Bramadhika (NIM: 235150701111031)  
**Program Studi**: Teknologi Informasi, Fakultas Ilmu Komputer, Universitas Brawijaya  

---
### Metodologi Evaluasi Pengetatan (Kriteria Baru)
Berdasarkan analisis hasil *Pilot Study* (16 Sel x 1 Replikasi), metodologi evaluasi Bab 3.4.5 Proposal Skripsi disempurnakan dengan **Multi-Dimensional SLA (Victim QoS + Host Protection)**:
1. **Victim Uptime ≥ 99.5%** (Ketersediaan layanan representatif normal sesuai komitmen AWS EC2 Single-Instance SLA).
2. **Victim Latency P95 ≤ 50.0 ms** (Ambang batas SLA microservice/API ringan industri, memperketat ambang 1000 ms agar relevan dengan baseline Nginx 1-2 ms).
3. **Host CPU Utilization (Fase Injeksi) ≤ 80.0%** (*Host Saturation Threshold* mengacu pada *AWS Well-Architected Framework* untuk mencegah kejenuhan kapasitas bersama).

### Skema Klasifikasi 3-Tingkat (*Tiers of Effectiveness*):
- 🟢 **Efektif Optimal**: Host aman (CPU ≤ 80%), Victim aman, Aggressor terbatasi via *CPU quota throttling* secara halus tanpa mati terbunuh.
- 🟡 **Efektif Ekstrem (OOM)**: Host aman, Victim aman, namun Aggressor dihentikan paksa oleh kernel *OOM Killer* karena kuota memori fisik terlampaui.
- 🔴 **Tidak Efektif**: Host mengalami saturasi komputasi (> 80%) dan/atau Victim terdegradasi akibat ketiadaan isolasi kuota CPU.


## 1. Inisialisasi Environment & Dependensi

In [ ]:
import json
import os
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import matplotlib
# Konfigurasi backend matplotlib agar aman di Jupyter maupun environment headless
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi gaya visualisasi publikasi
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

SLA_UPTIME_MIN_PCT = 99.5
SLA_LATENCY_MAX_MS = 50.0
SLA_HOST_CPU_MAX_PCT = 80.0

print("Environment siap: Pustaka data science berhasil dimuat dengan standar SLA pengetatan.")

## 2. Pemuatan Dataset Hasil Eksperimen (Auto-Discovery)
Membaca seluruh 16 berkas `combined_cell_*.json` dari direktori run terbaru di `results/`.

In [ ]:
def get_latest_results_dir(workspace_root: Path) -> Path:
    results_dir = workspace_root / "results"
    candidates = [p for p in results_dir.iterdir() if p.is_dir() and (p / "combined_cell_01_rep_01.json").exists()]
    if not candidates:
        raise FileNotFoundError("Tidak ditemukan folder run eksperimen yang valid di results/")
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

workspace = Path("..") if Path("..").resolve().name == "noisy-neighbor-research" else Path(".")
run_dir = get_latest_results_dir(workspace)
print(f"Memuat data dari direktori run: {run_dir.resolve()}")

records = []
for cell_id in range(1, 17):
    p = run_dir / f"combined_cell_{cell_id:02d}_rep_01.json"
    if not p.exists():
        continue
    with open(p, "r", encoding="utf-8") as f:
        d = json.load(f)
    
    cfg = d.get("config", {})
    tgt_base = d.get("target_metrics", {}).get("baseline", {})
    tgt_inj = d.get("target_metrics", {}).get("injection", {})
    cg = d.get("target_metrics", {}).get("cgroup_interventions", {})
    vic = d.get("jmeter_metrics", {}).get("victim_metrics", {})
    agg = d.get("jmeter_metrics", {}).get("aggressor_metrics", {})
    
    uptime = float(vic.get("victim_uptime_pct", 0.0))
    lat_p95 = float(vic.get("injection_latency_p95_ms", 0.0))
    host_cpu_inj = float(tgt_inj.get("host_cpu_pct_mean", 0.0))
    oom_killed = bool(cg.get("aggressor_oom_killed", 0))
    
    # Evaluasi Pengetatan Kriteria
    uptime_pass = uptime >= SLA_UPTIME_MIN_PCT
    lat_pass = lat_p95 <= SLA_LATENCY_MAX_MS
    cpu_pass = host_cpu_inj <= SLA_HOST_CPU_MAX_PCT
    
    is_eff = uptime_pass and lat_pass and cpu_pass
    status_biner = "Efektif" if is_eff else "Tidak Efektif"
    
    if not is_eff:
        status_tier = "Tidak Efektif"
        tier_code = 0
        alasan = f"Gagal: Host CPU Jenuh ({host_cpu_inj:.1f}% > 80%)" if not cpu_pass else "Gagal: SLA Pelanggaran"
    else:
        if oom_killed:
            status_tier = "Efektif Ekstrem (OOM)"
            tier_code = 1
            alasan = f"Lolos SLA (Agresor di-kill OOM t={cg.get('aggressor_oom_kill_second')}s)"
        else:
            status_tier = "Efektif Optimal"
            tier_code = 2
            alasan = "Lolos SLA (CPU Throttled halus tanpa OOM)"
    
    records.append({
        "cell_id": cell_id,
        "cell_name": cfg.get("cell_name", f"Sel {cell_id}"),
        "cpus": str(cfg.get("cpus")) if cfg.get("cpus") is not None else "None",
        "memory": str(cfg.get("memory")) if cfg.get("memory") is not None else "None",
        "host_cpu_base_pct": tgt_base.get("host_cpu_pct_mean", 0.0),
        "host_cpu_inj_pct": host_cpu_inj,
        "host_spare_mem_base_mb": tgt_base.get("host_spare_mem_mb_min", 0.0),
        "host_spare_mem_inj_mb": tgt_inj.get("host_spare_mem_mb_min", 0.0),
        "host_disk_tp_base_kbps": tgt_base.get("host_disk_throughput_kbps_mean", 0.0),
        "host_disk_tp_inj_kbps": tgt_inj.get("host_disk_throughput_kbps_mean", 0.0),
        "host_disk_await_base_ms": tgt_base.get("host_disk_await_ms_p95", 0.0),
        "host_disk_await_inj_ms": tgt_inj.get("host_disk_await_ms_p95", 0.0),
        "throttled_sec": round(cg.get("aggressor_throttled_usec_cumulative", 0) / 1_000_000.0, 2),
        "throttled_count": cg.get("aggressor_nr_throttled_cumulative", 0),
        "oom_kill_count": cg.get("aggressor_oom_kill_count", 0),
        "oom_killed": oom_killed,
        "oom_kill_second": cg.get("aggressor_oom_kill_second"),
        "victim_uptime_pct": uptime,
        "victim_lat_base_p95_ms": vic.get("baseline_latency_p95_ms", 0.0),
        "victim_lat_inj_p95_ms": lat_p95,
        "status": status_biner,
        "status_tier": status_tier,
        "tier_code": tier_code,
        "alasan": alasan,
    })

df = pd.DataFrame(records)
print(f"Dataset berhasil dimuat: {len(df)} sel pengujian.")
display(df[["cell_id", "cell_name", "cpus", "memory", "host_cpu_inj_pct", "victim_lat_inj_p95_ms", "throttled_sec", "oom_killed", "status", "status_tier", "alasan"]])

## 3. Rumusan Masalah 1: Penyebab & Dampak Noisy Neighbor pada Sel 1 Kontrol
Sel 1 beroperasi tanpa batas kuota CPU maupun memori (`cpus=None`, `memory=None`).
Evaluasi membuktikan apakah ketiadaan isolasi memicu kegagalan stabilitas host.

### 3.1 Visualisasi Time-Series Sel 1 Kontrol
Grafik ini menampilkan dinamika utilisasi CPU host yang melompat melewati garis toleransi saturasi host 80% (merah putus-putus) tepat setelah detik ke-10.

In [ ]:
ts_file = run_dir / "metrics_cell_01_rep_01.csv"
df_ts = pd.read_csv(ts_file)

fig, ax1 = plt.subplots(figsize=(11, 5.5))
color_cpu = "#d62728"
color_mem = "#1f77b4"

ax1.set_xlabel("Waktu Relatif (detik)", fontsize=12)
ax1.set_ylabel("Host CPU Utilization (%)", color=color_cpu, fontsize=12)
l1 = ax1.plot(df_ts["elapsed_s"], df_ts["host_cpu_pct"], color=color_cpu, linewidth=2.5, label="Host CPU (%)")
l_sat = ax1.axhline(y=80.0, color="#d62728", linestyle="--", linewidth=1.5, label="Batas Saturasi Host (80%)")
ax1.tick_params(axis="y", labelcolor=color_cpu)
ax1.set_ylim(-5, 105)

ax2 = ax1.twinx()
ax2.set_ylabel("Host Spare Memory (MB)", color=color_mem, fontsize=12)
l2 = ax2.plot(df_ts["elapsed_s"], df_ts["host_spare_mem_mb"], color=color_mem, linewidth=2.5, linestyle="--", label="Host Spare Mem (MB)")
ax2.tick_params(axis="y", labelcolor=color_mem)

l3 = plt.axvline(x=10.0, color="black", linestyle=":", linewidth=2, label="Awal Injeksi Beban (t=10s)")

lines = l1 + [l_sat] + l2 + [l3]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="center right", framealpha=0.9)
plt.title("Dinamika Sistem Sel 1 Kontrol: Kejenuhan CPU Host Melampaui Batas Toleransi 80%", fontsize=13, pad=15)
plt.tight_layout()
plt.show()

### 3.2 Tabel 3.4.1 Kuantifikasi Dampak Noisy Neighbor pada Sel 1 Kontrol
Kuantifikasi dampak dihitung membandingkan fase baseline (0–10s) terhadap fase injeksi (10–60s) menggunakan:
- **Rumus 3.2**: `Persentase Dampak = ((Nilai Injeksi - Nilai Baseline) / Nilai Baseline) * 100%`
- **Rumus 3.3**: `Persentase Dampak = ((Nilai Baseline - Nilai Injeksi) / Nilai Baseline) * 100%`

In [ ]:
s1 = df[df["cell_id"] == 1].iloc[0]

t341_data = [
    {"Variabel Terikat": "Host CPU Utilization (%)", "Metode Agregasi": "Rata-rata", "Baseline (0-10s)": s1["host_cpu_base_pct"], "Injeksi (10-60s)": s1["host_cpu_inj_pct"], "Rumus": "(3.2)", "Persentase Dampak (%)": round(((s1["host_cpu_inj_pct"] - s1["host_cpu_base_pct"]) / s1["host_cpu_base_pct"]) * 100.0, 2), "Status Pengetatan": "Terlampaui (91.2% > 80%)"},
    {"Variabel Terikat": "Host Spare Memory (MB)", "Metode Agregasi": "Minimum", "Baseline (0-10s)": s1["host_spare_mem_base_mb"], "Injeksi (10-60s)": s1["host_spare_mem_inj_mb"], "Rumus": "(3.3)", "Persentase Dampak (%)": round(((s1["host_spare_mem_base_mb"] - s1["host_spare_mem_inj_mb"]) / s1["host_spare_mem_base_mb"]) * 100.0, 2), "Status Pengetatan": "Terpangkas -43.9%"},
    {"Variabel Terikat": "Victim Latency P95 (ms)", "Metode Agregasi": "P95", "Baseline (0-10s)": s1["victim_lat_base_p95_ms"], "Injeksi (10-60s)": s1["victim_lat_inj_p95_ms"], "Rumus": "(3.2)", "Persentase Dampak (%)": round(((s1["victim_lat_inj_p95_ms"] - s1["victim_lat_base_p95_ms"]) / s1["victim_lat_base_p95_ms"]) * 100.0, 2), "Status Pengetatan": "Aman (2.0ms <= 50ms)"},
    {"Variabel Terikat": "Victim Uptime (%)", "Metode Agregasi": "Persentase", "Baseline (0-10s)": 100.0, "Injeksi (10-60s)": s1["victim_uptime_pct"], "Rumus": "(3.3)", "Persentase Dampak (%)": round(((100.0 - s1["victim_uptime_pct"]) / 100.0) * 100.0, 2), "Status Pengetatan": "Aman (100% >= 99.5%)"},
    {"Variabel Terikat": "Host Disk I/O Throughput (kB/s)", "Metode Agregasi": "Rata-rata", "Baseline (0-10s)": s1["host_disk_tp_base_kbps"], "Injeksi (10-60s)": s1["host_disk_tp_inj_kbps"], "Rumus": "(3.3)", "Persentase Dampak (%)": round(((s1["host_disk_tp_base_kbps"] - s1["host_disk_tp_inj_kbps"]) / s1["host_disk_tp_base_kbps"]) * 100.0, 2), "Status Pengetatan": "Aktivitas Rendah"},
    {"Variabel Terikat": "Host Disk I/O Latency (ms)", "Metode Agregasi": "P95", "Baseline (0-10s)": s1["host_disk_await_base_ms"], "Injeksi (10-60s)": s1["host_disk_await_inj_ms"], "Rumus": "(3.2)", "Persentase Dampak (%)": round(((s1["host_disk_await_inj_ms"] - s1["host_disk_await_base_ms"]) / s1["host_disk_await_base_ms"]) * 100.0, 2) if s1["host_disk_await_base_ms"] > 0 else 0.0, "Status Pengetatan": "Aman (0.0ms)"},
]

df_t341 = pd.DataFrame(t341_data)
display(df_t341)

## 4. Rumusan Masalah 2: Uji Efektivitas 16 Sel Konfigurasi Cgroups
Evaluasi efektivitas 16 kombinasi faktorial penuh dengan kriteria pengetatan baru (Host CPU ≤ 80%, Latensi P95 ≤ 50ms, Uptime ≥ 99.5%).

In [ ]:
s1 = df[df["cell_id"] == 1].iloc[0]
t343_rows = []

for _, r in df.iterrows():
    eff_cpu = ((s1["host_cpu_inj_pct"] - r["host_cpu_inj_pct"]) / s1["host_cpu_inj_pct"]) * 100.0
    eff_mem = ((r["host_spare_mem_inj_mb"] - s1["host_spare_mem_inj_mb"]) / s1["host_spare_mem_inj_mb"]) * 100.0
    
    t343_rows.append({
        "Sel": f"Sel {r.cell_id}",
        "Limit CPU": r.cpus,
        "Limit Memori": r.memory,
        "Host CPU (%)": r.host_cpu_inj_pct,
        "Efektivitas CPU (%)": round(eff_cpu, 2),
        "Host Spare Mem (MB)": r.host_spare_mem_inj_mb,
        "Efektivitas Memori (%)": round(eff_mem, 2),
        "Throttling (s)": r.throttled_sec,
        "OOM Killed": "Ya" if r.oom_killed else "Tidak",
        "Victim Latency P95 (ms)": r.victim_lat_inj_p95_ms,
        "Status Biner": r.status,
        "Status 3-Tier": r.status_tier,
        "Justifikasi Ilmiah": r.alasan
    })

df_t343 = pd.DataFrame(t343_rows)
display(df_t343)

## 5. Visualisasi 4x4 Heatmap Grid Faktorial Penuh

In [ ]:
CPU_LEVELS = ["None", "1.5", "1.0", "0.5"]
CPU_LABELS = ["Tanpa Limit\n(2.0 vCPU)", "1.5 vCPU", "1.0 vCPU", "0.5 vCPU"]
MEM_LEVELS = ["None", "768m", "512m", "256m"]
MEM_LABELS = ["Tanpa Limit\n(Maks)", "768 MB", "512 MB", "256 MB"]

def build_grid(df: pd.DataFrame, metric: str) -> np.ndarray:
    grid = np.zeros((4, 4))
    for i, c in enumerate(CPU_LEVELS):
        for j, m in enumerate(MEM_LEVELS):
            sub = df[(df["cpus"] == c) & (df["memory"] == m)]
            if not sub.empty:
                val = sub.iloc[0].get(metric, np.nan)
                if pd.isna(val) and metric == "oom_kill_count" and "oom_killed" in sub.columns:
                    val = 1.0 if sub.iloc[0]["oom_killed"] else 0.0
                grid[i, j] = val
            else:
                grid[i, j] = np.nan
    return grid

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Host CPU Utilization
g_cpu = build_grid(df, "host_cpu_inj_pct")
sns.heatmap(g_cpu, annot=True, fmt=".1f", cmap="YlOrRd", xticklabels=MEM_LABELS, yticklabels=CPU_LABELS, cbar_kws={"label": "CPU (%)"}, ax=axes[0, 0], vmin=0, vmax=100)
axes[0, 0].set_title("A. Host CPU Utilization (%) (Batas Aman Maks: 80%)", fontsize=13, pad=10)
axes[0, 0].set_xlabel("Batas Memori Aggressor")
axes[0, 0].set_ylabel("Batas CPU Aggressor")

# 2. Host Spare Memory
g_mem = build_grid(df, "host_spare_mem_inj_mb")
sns.heatmap(g_mem, annot=True, fmt=".0f", cmap="Blues", xticklabels=MEM_LABELS, yticklabels=CPU_LABELS, cbar_kws={"label": "Spare Mem (MB)"}, ax=axes[0, 1])
axes[0, 1].set_title("B. Host Spare Memory Minimum (MB) Fase Injeksi", fontsize=13, pad=10)
axes[0, 1].set_xlabel("Batas Memori Aggressor")
axes[0, 1].set_ylabel("Batas CPU Aggressor")

# 3. Aggressor Throttling (detik)
g_throt = build_grid(df, "throttled_sec")
sns.heatmap(g_throt, annot=True, fmt=".2f", cmap="Purples", xticklabels=MEM_LABELS, yticklabels=CPU_LABELS, cbar_kws={"label": "Throttling (detik)"}, ax=axes[1, 0])
axes[1, 0].set_title("C. Aggressor CPU Throttling Duration (detik)", fontsize=13, pad=10)
axes[1, 0].set_xlabel("Batas Memori Aggressor")
axes[1, 0].set_ylabel("Batas CPU Aggressor")

# 4. Aggressor OOM Kill Occurrences
g_oom = build_grid(df, "oom_kill_count")
sns.heatmap(g_oom, annot=True, fmt=".0f", cmap="Reds", xticklabels=MEM_LABELS, yticklabels=CPU_LABELS, cbar_kws={"label": "OOM Count"}, ax=axes[1, 1])
axes[1, 1].set_title("D. Intervensi OOM Killer pada Aggressor (1=Killed)", fontsize=13, pad=10)
axes[1, 1].set_xlabel("Batas Memori Aggressor")
axes[1, 1].set_ylabel("Batas CPU Aggressor")

plt.tight_layout()
plt.show()

### 5.2 Matriks 4x4 Status Efektivitas Pengetatan (Biner & 3-Tier)
Memperlihatkan kegagalan Sel 1 dan Sel 2 (merah) akibat kejenuhan CPU host (>80%), serta efektivitas konfigurasi cgroups (hijau/oranye).

In [ ]:
fig, (ax_bin, ax_tier) = plt.subplots(1, 2, figsize=(20, 8))

# Matriks Biner (Efektif vs Tidak Efektif)
grid_bin = np.zeros((4, 4))
annots_bin = np.empty((4, 4), dtype=object)
for i, c in enumerate(CPU_LEVELS):
    for j, m in enumerate(MEM_LEVELS):
        sub = df[(df["cpus"] == c) & (df["memory"] == m)]
        if not sub.empty:
            r = sub.iloc[0]
            is_eff = (r["status"] == "Efektif")
            grid_bin[i, j] = 1 if is_eff else 0
            st_txt = "EFEKTIF" if is_eff else "TIDAK EFEKTIF"
            annots_bin[i, j] = f"Sel {r['cell_id']}\n{st_txt}\n(CPU {r['host_cpu_inj_pct']:.1f}%)"

sns.heatmap(grid_bin, annot=annots_bin, fmt="", cmap=sns.color_palette(["#d62728", "#2ca02c"]), cbar=False, xticklabels=MEM_LABELS, yticklabels=CPU_LABELS, ax=ax_bin, linewidths=2, linecolor="white")
ax_bin.set_title("Matriks Biner: Efektif vs Tidak Efektif\n(Kriteria: Uptime ≥ 99.5%, Latensi P95 ≤ 50ms, Host CPU ≤ 80%)", fontsize=13, pad=15)
ax_bin.set_xlabel("Batas Memori Aggressor")
ax_bin.set_ylabel("Batas CPU Aggressor")

# Matriks 3-Tier
grid_tier = np.zeros((4, 4))
annots_tier = np.empty((4, 4), dtype=object)
for i, c in enumerate(CPU_LEVELS):
    for j, m in enumerate(MEM_LEVELS):
        sub = df[(df["cpus"] == c) & (df["memory"] == m)]
        if not sub.empty:
            r = sub.iloc[0]
            t_code = r["tier_code"]
            grid_tier[i, j] = t_code
            t_name = "Optimal" if t_code == 2 else ("OOM Intervened" if t_code == 1 else "TIDAK EFEKTIF")
            annots_tier[i, j] = f"Sel {r['cell_id']}\n{t_name}\n(CPU {r['host_cpu_inj_pct']:.1f}%)"

sns.heatmap(grid_tier, annot=annots_tier, fmt="", cmap=sns.color_palette(["#d62728", "#ff7f0e", "#2ca02c"]), cbar=False, xticklabels=MEM_LABELS, yticklabels=CPU_LABELS, ax=ax_tier, linewidths=2, linecolor="white")
ax_tier.set_title("Matriks 3-Tingkat Efektivitas Isolasi Cgroups\n(Merah = Gagal Host CPU >80% | Oranye = OOM Killer | Hijau = Graceful Throttling)", fontsize=13, pad=15)
ax_tier.set_xlabel("Batas Memori Aggressor")
ax_tier.set_ylabel("Batas CPU Aggressor")

plt.tight_layout()
plt.show()

## 6. Analisis Cross-Resource Side-Effects (Disk I/O)
Menganalisis throughput dan latensi disk antar sel pengujian.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df))
width = 0.35

ax.bar(x - width/2, df["host_disk_tp_base_kbps"], width, label="Baseline (0-10s)", color="#1f77b4")
ax.bar(x + width/2, df["host_disk_tp_inj_kbps"], width, label="Fase Injeksi (10-60s)", color="#ff7f0e")

ax.set_xticks(x)
ax.set_xticklabels([f"S{cid}" for cid in df["cell_id"]], fontsize=10)
ax.set_xlabel("Sel Pengujian (Sel 1 s/d Sel 16)", fontsize=12)
ax.set_ylabel("Host Disk Throughput (kB/s)", fontsize=12)
ax.set_title("Perbandingan Disk Throughput Baseline vs Injeksi antar Sel Pengujian", fontsize=13, pad=12)
ax.legend(framealpha=0.9)
plt.tight_layout()
plt.show()

## 7. Sintesis Temuan Pilot Study & Kesimpulan Ilmiah

### Temuan Utama Berdasarkan Kriteria Pengetatan:
1. **Pembuktian Dampak Noisy Neighbor (RM 1)**:
   - Pada **Sel 1 (Kontrol)** dan **Sel 2**, ketiadaan batas kuota CPU menyebabkan penggunaan CPU host melonjak hingga **91.15% dan 91.52%**, melanggar batas toleransi saturasi host (80.0%).
   - Keduanya resmi terklasifikasi sebagai **TIDAK EFEKTIF**, membuktikan bahwa *noisy neighbor* merusak stabilitas lingkungan *multi-tenant* jika CPU tidak diisolasi.

2. **Keberhasilan Isolasi Kuota CPU (RM 2)**:
   - Penerapan batas CPU terbukti efektif mengendalikan utilisasi host:
     - `cpus=1.5`: Host CPU tertahan di ~70.3% (throttled ~22-23 detik) -> **EFEKTIF OPTIMAL**.
     - `cpus=1.0`: Host CPU tertahan di ~47.5% (throttled ~46-47 detik) -> **EFEKTIF OPTIMAL**.
     - `cpus=0.5`: Host CPU tertahan di ~24.8% (throttled ~70 detik) -> **EFEKTIF OPTIMAL**.
   - Mekanisme CPU quota throttling bekerja secara *graceful*, menahan beban agresor tanpa menghentikan aplikasinya.

3. **Perilaku Deterministik OOM Killer pada Limit Memori**:
   - Pada batas memori ketat (`memory=512m` dan `256m`), kernel Linux memicu OOM Killer pada detik ~15-30s, menghentikan container agresor secara paksa -> **EFEKTIF EKSTREM (OOM)**.
   - Ini mendokumentasikan *trade-off* nyata: batas memori yang terlalu ketat melindungi host tetapi mengorbankan kelangsungan layanan penyewa.